# What Memory Is

**Prerequisites:** from `01_foundations`: `02_conversations.ipynb`, `03_tokens_and_context.ipynb`. From `02_agent_runtime`: `02_the_agent_loop.ipynb`

Chapters 2 and 3 built an agent that reasons in a loop and acts on the world. Neither keeps anything once the loop returns: memory lasts exactly as long as the variable holding it.

That sounds like a missing feature. It is not. The agent already *has* memory — it has had it since chapter 1 — and this notebook is mostly about learning to see it. `Conversation.messages` is memory. The scratchpad an agent scribbles a plan on is memory. Chapter 1's `key_facts` is memory. None of them are called that, and none of them survive a restart.

**Two problems get muddled together here, and this notebook separates them.**

The first is a **conversation that has to survive between turns**. That splits in two, and the first half of this notebook builds both: a durable copy for someone who comes back tomorrow, and a hot one for the worker answering their next message thirty seconds from now. Different stores, for reasons that only show up when you look at what ends each of them.

The second is **facts worth keeping long after the conversation they came from is over**. That one is not settled, and the second half of this notebook works out why it needs something neither a file nor a conversation store can do — which is what the rest of the chapter is about.

**Nothing here needs infrastructure.** A list, a dict, a JSON file, and an in-process stand-in for Redis that behaves like the real thing.

**By the end you'll have:**
- Watched a second session meet you as a stranger, and seen that nothing broke — there was simply nowhere for the first session to go
- Seen that the whole of an agent's memory is the list it sends, by printing it
- Carried a conversation across the gap with a file, then with Redis: a second worker picking it up from the store, and a lease that expires on its own with no cleanup job to write
- Built `Scratchpad`: memory the agent can revise, which a conversation can never be
- Found the point where memory runs out *inside* a single session, using chapter 1's `trim_to_budget`
- Seen chapter 1's `key_facts` for what it is — the repo's first long-term memory, living in short-term storage, and measured what extraction buys
- Arrived at the thing a file cannot do for facts, and written the recall questions that force a store

## Setup

`memory.py` sits one folder up. Chapter 1's `chat.py` and `llm.py` come from their own folder — this notebook leans on them heavily, because most of what it examines was already built there.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

from chat import Conversation, count_tokens, trim_to_budget
from memory import Scratchpad, load_session, save_session

SESSION_FILE = Path("session.json")

## The agent that meets you twice

Two sessions, a few minutes apart. In the first one, you mention something about yourself in passing — not as a question, just as context.

In [2]:
first = Conversation(system="You are a terse assistant. Two sentences at most.")

first.send("Quick context before we start: I'm in Asia/Kolkata, and I'd rather see examples in Python than JavaScript.")

"Got it—I'll tailor examples to Python and use the Asia/Kolkata timezone. What would you like to see?"

It heard you. Ask it now and it will tell you your timezone without hesitating, because the sentence you typed is still sitting in the same list it is about to send to the API.

Now the session ends. A new one starts — which in a notebook means nothing more dramatic than constructing a second `Conversation`.

In [3]:
second = Conversation(system="You are a terse assistant. Two sentences at most.")

second.send("What timezone am I in, and should I expect examples in Python or JavaScript?")

'I don’t have enough information to determine your timezone. I can provide examples in either Python or JavaScript—just let me know which you prefer.'

It has no idea, and it is right not to.

Nothing failed here. There is no bug to find, no exception, no dropped request. `first.messages` is still in memory in this notebook — it is just not in `second.messages`, and `second.messages` is the entire world as far as that second call is concerned. The model saw a system message and one question.

That is the actual shape of the problem: **an agent's memory is whatever is in the list it sends, and nothing else.**

## Handing a session to the next one

Everything so far lives and dies in one process. The smallest thing that changes that is a file.

`save_session` writes the conversation out as JSON — the messages themselves, in the shape the API expects them back.

In [5]:
payload = save_session(SESSION_FILE, first)

print(SESSION_FILE.read_text(encoding="utf-8")[:280], "...")
print()
print("messages saved:", len(payload["messages"]))

{
  "messages": [
    {
      "role": "system",
      "content": "You are a terse assistant. Two sentences at most."
    },
    {
      "role": "user",
      "content": "Quick context before we start: I'm in Asia/Kolkata, and I'd rather see examples in Python than JavaScript."
   ...

messages saved: 3


Now the second session again — a new `Conversation`, exactly as before, except that this time it loads first.

In [6]:
third = Conversation(system="You are a terse assistant. Two sentences at most.")
load_session(SESSION_FILE, third)

print("loaded messages:", len(third.messages))
print("first one back  :", third.messages[1]["content"][:64])

loaded messages: 3
first one back  : Quick context before we start: I'm in Asia/Kolkata, and I'd rath


In [7]:
third.send("What timezone am I in, and should I expect examples in Python or JavaScript?")

"You're in the **Asia/Kolkata** timezone, and I'll provide examples in **Python** as you prefer."

The question that failed at the top of this notebook now answers.

Nothing clever happened. The second session is reading the *same conversation* the first one had — the file carried it across, and the model sees the sentence you typed at the start exactly as if you had never left. Session memory does not need to be summarised or distilled to work; it needs to be somewhere both halves of the conversation can reach.

### The other half: the session that is still live

What the file just did was hold a conversation for someone who **came back later**. That is one
problem. There is a second one it handles badly, and a running system has both:

|  | the file | the live session |
|---|---|---|
| when it is read | once, when they return | on every single turn |
| who reads it | whoever serves them next | whichever worker takes the next message |
| how long it must last | until they come back — days, months | minutes |
| what ends it | nothing; you keep it | they walk away, and nothing marks that |

A file is fine for the first column and wrong for the second: it is on one machine's disk, and
nothing ever tells it a conversation is over. **Redis** is the usual answer for the second, and
the difference is worth doing rather than describing — so here are the same two moves again, one
worker writing and another picking up.

`fakeredis` is used so this notebook still runs with nothing installed. It implements the real
command surface, so against a live server the only lines that change are the connections:

```python
worker_a = redis.Redis(host="redis.internal", port=6379, decode_responses=True)
worker_b = redis.Redis(host="redis.internal", port=6379, decode_responses=True)
```

In [8]:
import json
import time

import fakeredis

# one store, two clients -- the same shape as two processes talking to one Redis
store = fakeredis.FakeServer()
worker_a = fakeredis.FakeStrictRedis(server=store, decode_responses=True)
worker_b = fakeredis.FakeStrictRedis(server=store, decode_responses=True)

payload = {"messages": [dict(m) for m in first.messages]}

# ex= is the part a file has no equivalent for: an hour from now this deletes itself
worker_a.set("session:abc123", json.dumps(payload), ex=3600)

print("worker A stored:", worker_a.keys("session:*"))
print("lease          :", worker_a.ttl("session:abc123"), "seconds")

worker A stored: ['session:abc123']
lease          : 3600 seconds


Now the second worker — exactly what `third` did with the file a moment ago, except that this
one is a different connection rather than a different `Conversation` object.

In [9]:
raw = worker_b.get("session:abc123")
recovered = json.loads(raw)

print("worker B loaded:", len(recovered["messages"]), "messages")
print("and can answer from:", recovered["messages"][1]["content"][:58])

# every turn renews the lease, so a conversation still in use never expires
worker_b.expire("session:abc123", 3600)
print()
print("worker B renewed the lease; worker A sees:",
      worker_a.ttl("session:abc123"), "seconds")

worker B loaded: 3 messages
and can answer from: Quick context before we start: I'm in Asia/Kolkata, and I'

worker B renewed the lease; worker A sees: 3600 seconds


Worker B never touched worker A's memory or its disk. It asked the store, and the store had it —
which is the whole point, because in production those are two processes on two machines, and a
conversation does not get to care which one answers turn five.

Note the renewal on the last line. Every turn pushes the expiry back, so a conversation in active
use never dies, and one that stopped gets exactly one hour of grace. That is a single call, and
it is the behaviour you would otherwise be building by hand over files.

Now the part the file genuinely cannot do.

In [10]:
# the same lease, one second long, so the expiry is watchable
worker_a.set("session:temp", json.dumps(payload), ex=1)

for moment in ["just written", "0.4s later", "0.8s later", "1.2s later"]:
    alive = bool(worker_b.exists("session:temp"))
    print(f"{moment:>12}  ->  worker B sees it: {str(alive):<5}  ttl: {worker_b.ttl('session:temp'):>2}")
    time.sleep(0.4)

print()
print("meanwhile the real session:", worker_a.ttl("session:abc123"), "seconds left")

just written  ->  worker B sees it: True   ttl:  1


  0.4s later  ->  worker B sees it: True   ttl:  1


  0.8s later  ->  worker B sees it: True   ttl:  0


  1.2s later  ->  worker B sees it: False  ttl: -2



meanwhile the real session: 3597 seconds left


Nobody deleted it. The conversation simply stopped mattering, and the store knew that.

**Conversations are abandoned far more often than they are ended** — someone closes a tab, and
nothing ever marks that session finished. With files, dealing with that is a job you have to
write and then remember to run: something that walks the directory, decides which sessions are
old enough, and deletes them. Nobody writes that job for Redis. Expiry is a property of the key,
enforced by the store, and there is no sweeper to build, schedule or debug.

**And that is also the limit.** Run the file section's demo against a one-hour lease and come
back tomorrow, and there is nothing to load: the property that cleans up abandoned sessions is
the same property that loses resumable ones. The lease is not a detail to tune around — it is
what you are buying.

So the two are not a worse and a better version of one thing. A production system runs both:

- **Redis** holds the conversation while it is live. Read every turn, shared across workers,
  gone shortly after the last message.
- **A durable store** holds the transcript for anyone who comes back — Postgres rather than a
  JSON file, but doing the file's job, not Redis's.

Which one a message goes to is decided by the same question as everything else in this chapter:
what will be asked of it, and when. "Answer the next turn" and "let them scroll back next month"
are different questions, and reaching for one store to serve both means picking which of them to
do badly.

## A scratchpad: memory that can be revised

There is one more kind of short-term memory, and it exists because of something a conversation structurally cannot do.

`messages` is append-only. A message that has been sent cannot be taken back, so when a plan changes, both versions stay in the list and the model has to work out which one is current — usually from position, which is one of the things it is least reliable at. Chapter 2's `replan` walks straight into this.

In [11]:
planning = Conversation(system="You are a planner.")
planning.messages += [
    {"role": "assistant", "content": "Plan: 1) read the config file  2) call the billing API  3) email the summary"},
    {"role": "user", "content": "The billing API is down for maintenance today."},
    {"role": "assistant", "content": "Plan: 1) read the config file  2) read yesterday's cached billing export  3) email the summary"},
]

for message in planning.messages:
    print(f"{message['role']:>9}: {message['content'][:78]}")

   system: You are a planner.
assistant: Plan: 1) read the config file  2) call the billing API  3) email the summary
     user: The billing API is down for maintenance today.
assistant: Plan: 1) read the config file  2) read yesterday's cached billing export  3) e


Two plans, both present, both phrased with equal confidence. The only thing marking the second as current is that it comes later.

A scratchpad is keyed instead of appended, so writing to `plan` twice *replaces* rather than accumulates.

In [12]:
pad = Scratchpad()

pad.write("plan", "1) read the config  2) call the billing API  3) email the summary")
pad.write("plan", "1) read the config  2) read yesterday's cached export  3) email the summary")
pad.write("attempts", 2)

print(pad.render())
print()
print("entries:", len(pad))

Scratchpad:
- plan: 1) read the config  2) read yesterday's cached export  3) email the summary
- attempts: 2

entries: 2


One plan, and it is the current one. The pad stays the same size however many times the agent changes its mind, which is the property that matters when it goes into a prompt on every turn.

`Scratchpad.erase()` is deliberately not called `forget`. Forgetting is a real operation on long-term memory, with consequences — a fact that is wrong, a fact that has expired, a fact that contradicts another — and it gets its own treatment in notebook 3. Clearing a key on a pad that dies at the end of the session is not that.

## Memory runs out before the session does

The second session failing is the obvious problem. The less obvious one is that memory runs out *within* a session, and chapter 1 already built the thing that makes it happen.

Every message is re-sent on every call, so a conversation costs more the longer it runs, and eventually it does not fit. `trim_to_budget` from `01_foundations/chat.py` handles that by dropping whole turns from the front — oldest first, keeping tool round-trips intact.

Here is a session where something worth knowing was said early, followed by ordinary work.

In [13]:
history = [
    {"role": "system", "content": "You are a terse assistant."},
    {"role": "user", "content": "I'm in Asia/Kolkata, and I'd rather see examples in Python than JavaScript."},
    {"role": "assistant", "content": "Noted -- Asia/Kolkata, Python examples."},
    {"role": "user", "content": "How should I lay out a Python package that has both a library and a CLI?"},
    {"role": "assistant", "content": "Put the library under src/yourpkg/ and the CLI entry point in src/yourpkg/__main__.py, wired up through the project's console_scripts entry point so it installs as a real command."},
    {"role": "user", "content": "And where do the tests go, next to the code or in a top-level folder?"},
    {"role": "assistant", "content": "A top-level tests/ folder, mirroring the package layout. Keeping them outside the package means the installed wheel does not ship them, and it forces you to import the package the way a user would."},
    {"role": "user", "content": "What about configuration -- environment variables or a config file?"},
    {"role": "assistant", "content": "Environment variables for anything that differs per machine or is secret, a checked-in config file for defaults that are the same everywhere. A .env file loaded at startup gives you both without mixing them."},
]

print("messages:", len(history), "| tokens:", sum(count_tokens(m["content"]) for m in history))

messages: 9 | tokens: 202


In [14]:
kept = trim_to_budget(history, max_tokens=250)

print("kept", len(kept), "of", len(history), "messages\n")
for message in kept:
    print(f"{message['role']:>9}: {message['content'][:70]}")

kept 5 of 9 messages

   system: You are a terse assistant.
     user: And where do the tests go, next to the code or in a top-level folder?
assistant: A top-level tests/ folder, mirroring the package layout. Keeping them 
     user: What about configuration -- environment variables or a config file?
assistant: Environment variables for anything that differs per machine or is secr


The timezone is gone.

Nothing malfunctioned — the trim did exactly what it was asked to. But notice *what kind* of thing it dropped. The turns that survived are answers to questions already answered; the turn that was discarded is the only one that said something durable about the person on the other end. Trimming by age throws away whichever facts happen to be old, and a fact stated early is a fact stated once.

That is the case for keeping something on the way out.

## Chapter 1 already built long-term memory

`Conversation.trim(remember=True)` reads the turns it is about to drop, pulls out anything worth keeping, and appends it to `conversation.notes`. It is a handful of lines in `chat.py` and it is easy to walk past, but it is the first long-term memory in this repo: a fact, extracted from an event, deliberately kept after the event is gone.

This cell makes a real model call — the extraction is `extract()` from `llm.py`.

In [15]:
session = Conversation(system="You are a terse assistant.")
session.messages = list(history)

session.trim(max_tokens=250, remember=True)

print("notes:", session.notes)
print("messages kept:", len(session.messages))
print()
print("the 9 messages cost:", sum(count_tokens(m["content"]) for m in history), "tokens")
print("the notes cost     :", sum(count_tokens(n) for n in session.notes), "tokens")

notes: ['User is in timezone Asia/Kolkata', 'User prefers Python examples over JavaScript']
messages kept: 5

the 9 messages cost: 202 tokens
the notes cost     : 13 tokens


Look at the two numbers the cell printed. The nine messages cost 202 tokens; the notes distilled out of them cost a small fraction of that. Both can answer "what timezone is this user in".

The reason that gap matters is that **memory is re-sent on every call**. Keeping the conversation means paying its full cost on this turn, and on the next one, and on every turn after that, forever. Keeping the notes means paying the smaller number instead — for the same specific detail. Extraction is not tidying up: it is the difference between paying to carry what happened and paying to carry what specific details you will actually need next time.

But look at where the fact went. `session.notes` is a Python list on an object in this kernel. It has exactly the same lifetime as `session.messages` — it dies with the process and this is exactly why we need to persist the specific details to suitable stores that can serve by handing few of those relevant ones based on query.

## The questions the rest of the chapter owes an answer to

Each of these is a question this file cannot answer. They are what decides, in notebook 2, where each memory gets written in the first place:

1. **"What is this user's timezone?"** — you know exactly what you are looking for. A key, and a dict; no similarity search, no embedding, no index. Most long-term memory in a real agent is this, and reaching for a vector database here makes the answer *approximate* for no benefit.
2. **"Which deployment failed most recently?"** — you know the shape but not the answer, and the answer depends on ordering and filtering. A file cannot sort; a list in memory can, until it does not fit.
3. **"Where did that `PGRST204` error come from?"** — an exact identifier. Nothing but the literal token will do, and near-matches are worse than useless.
4. **"Has this happened before?"** — all you have is a description of what is going wrong now: *checkout is timing out and the database seems to be blocking*. There is no key, and no field to filter on either, because the sentence gives you a **symptom** rather than a category. This is the question that forces embeddings, and the only one of the five that genuinely does.
5. **"Who else touched this service, and what did they decide?"** — the answer is not in any one memory; it is in how several of them connect.

Notebook 2 takes these in turn, and each store arrives because the previous one cannot answer the next question.

Two things also came up here that the later notebooks pick up directly: chapter 1's `notes` are long-term content in short-term storage, and getting a fact *stored* turned out to be a different job from getting it *seen*.

In [16]:
SESSION_FILE.unlink(missing_ok=True)

## What you built

✓ Watched a second session meet you as a stranger, and established that nothing broke doing it — `first.messages` and `second.messages` are two different lists, and the second call only ever sees its own

✓ Printed that list, because the claim it supports is the whole premise: an agent's memory is whatever is in the messages it sends, and nothing else

✓ Carried the conversation across the gap with `save_session` and `load_session`, and answered the opening question from the *same conversation* rather than a summary of it — session memory does not need distilling to work, it needs somewhere both halves can reach

✓ Did it again against **Redis**, where session state actually lives: one store, two clients, and worker B answering from what worker A wrote without touching its memory or its disk

✓ Watched a key expire on its own, and named what that replaces — with files, deciding which sessions are stale is a job you write and then remember to run; with a lease there is no sweeper to build, schedule or debug

✓ Renewed a lease in one call, so a conversation in active use never dies and an abandoned one gets exactly its hour

✓ Built `Scratchpad` in `memory.py`: keyed rather than appended, so a revised plan replaces the old one instead of leaving both in the context for the model to disambiguate by position

✓ Kept `erase()` deliberately distinct from *forgetting*, which is an operation on long-term memory with real consequences and belongs to notebook 3

✓ Ran chapter 1's `trim_to_budget` and watched it drop the only turn that said something durable about the user, because trimming by age discards whichever facts happen to be old

✓ Used `Conversation.trim(remember=True)` and measured what extraction buys: 202 tokens of conversation reduced to a handful of tokens of fact, both answering the same question, on a bill you pay every single turn

✓ Established two things about that extraction the rest of the chapter depends on — it is a model's opinion and varies between runs, and the prompt deciding what to keep is deciding what can ever be recalled

✓ Found where the two halves part company: a conversation store holds one blob under one name, which is exactly right for a conversation and useless for facts, because recall becomes all-or-nothing

✓ Written the recall questions that force a store — each one a query a file cannot serve

**Next:** `02_stores.ipynb` — those questions, taken one at a time, deciding where each memory belongs from what will be asked of it.